In [ ]:
import torch

# A better loss for classification

Cross-entropy measures how well a predicted probability distribution ($p$) matches a target
  distribution ($q$).

Our target distribution is one-hot: it assigns a probability of one to the target class ($y$) and zero to every other class:

$$
q_{j}=\mathbf{1}[j=y].
$$

This simplifies cross-entropy to minimising the negative log-likelihood of the output for the target class ($p_{y}$):

$$
  \mathcal{L}=-\log p_{y}.
$$

Minimising this loss with gradient descent, is equivalent to maximising the likelihood of the output for the target class.


To convert the output layer of a neural network $z$ to a categorical probability distribution, we can use the softmax function:

$$
  \mathcal{L}=

  -
  \log
  \left(
  \frac{\exp(z_{y})}
  {\sum_{j=1}^{C}\exp(z_{j})}
  \right),
$$
where $C$ is the number of classes. The loss will be lower when the model asigns more probability to the correct class.

And adding a rank to account for a mini-batch:

$$
  \mathcal{L}=

  -\frac{1}{N}\sum_{i=1}^{N}
  \log
  \left(
  \frac{\exp(z_{i,y_i})}
  {\sum_{j=1}^{C}\exp(z_{ij})}
  \right).
$$

where $N$ is the size of the mini-batch.


In [ ]:
model_output = torch.tensor([[1.0, 2.0, 3.0], [10.0, 10.0, 10.0]])  # y_hat_i
target = torch.tensor([1, 0])  # y_i

In [ ]:
# Before:
target_one_hot = torch.nn.functional.one_hot(target, num_classes=3).float()
loss = torch.nn.functional.mse_loss(model_output, target_one_hot)
loss

In [ ]:
# After:
loss = torch.nn.functional.cross_entropy(model_output, target)
loss

# Convolutional neural network

Involves convolving a set of learnable kernels/filters over an input.

Below is a single 2D kernel; it may learn local patterns such as edges, textures, and shapes during training.

In [ ]:
from IPython.display import Image, display

display(Image(filename='./02-mnist-example/images/kernel.gif'))

With MNIST, we dealt with images with one channel (greyscale images). 

With coloured images, we have three channels (red, green, blue).

So our 2D CNN layer typically have four parameters:
- Number of input channels
- Kernel height
- Kernal width
- Number of filters/kernels/output channels


E.g., this 2D CNN layer has:
- Three input channels (red, green, and blue of the image)
- A kernel height of four
- A kernel height of four
- Two filters/kernels/output channels

In [ ]:
display(Image(filename='./02-mnist-example/images/kernel_2.gif'))

In [ ]:
from torch import nn

conv_2d_layer = nn.Conv2d(in_channels=3, out_channels=2, kernel_size=4)

In [ ]:
from PIL import Image
from torchvision.transforms import ToTensor

image_1_path = "02-mnist-example/images/bird6.png"
image_2_path = "02-mnist-example/images/airplane1.png"

image_1 = Image.open(image_1_path)
image_2 = Image.open(image_2_path)

image_tensor_1 = ToTensor()(image_1)
image_tensor_2 = ToTensor()(image_2)

display(image_1)
display(image_2)

In [ ]:
display(image_1.resize((320, 320)))
display(image_2.resize((320, 320)))

In [ ]:
# Conv2d expects (batch, channels, height, width).
image_batch = torch.stack([image_tensor_1, image_tensor_2], dim=0)

output = conv_2d_layer(image_batch)

print("Input:", image_batch.shape)
print("Output:", output.shape)

Lets make our own CNN:

![](./images/CNN.png)

In [ ]:
from torch import nn

class BasicCNN(nn.Module):
    def __init__(
        self,
        image_height: int,
        image_width: int,
        num_classes: int = 10,
    ):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        flattened_size = 32 * (image_height // 4) * (image_width // 4)

        self.flatten = nn.Flatten()
        self.classifier = nn.Linear(flattened_size, num_classes)  # We need to flatten the output for the linear layer.


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.flatten(x)
        return self.classifier(x)

In [ ]:
model = BasicCNN(num_classes=10, image_height=32, image_width=32)
model

In [ ]:
logits = model(image_batch)
logits.shape

# Momentum

In gradient descent, the initial values of learnable parameters can determine which local minimum the optimisation converges to.

This is especially true for non-convex loss landscapes, such as those in neural networks.

There are several methods to more consistently reach lower minima:
  - **Improved optimisation**: Use momentum, adaptive
    optimisers, learning-rate schedules, warm-up, or
    multiple random restarts.

  - **Architecture design**: Residual connections,
    normalisation layers, and suitable activation
    functions can make the loss landscape easier to
    optimize.

  - **Careful initialisation**: Methods such as Xavier or He
    initialization improve gradient flow and reduce
    sensitivity to starting values.

  - **Data augmentation**: This does not directly simplify
    the optimisation landscape. Instead, it regularises
    the model and encourages solutions that generalise
    better.

“Lower minimum” does not necessarily mean “better model”.

This could mean that you have overfitted to your training data.

This is why we use a validation set with examples independant of the training set.

Hence, we want minima that results in strong validation performance.

In [ ]:
from IPython.display import display, Image

display(Image(filename='./02-mnist-example/images/gradient_descent.gif'))

Momentum is one method to avoid getting stuck in local minima.

Our standard stochastic gradient descent optimiser was:

  $$
  w_{t+1}
  =
  w_t
  -
  \eta \frac{\partial \mathcal{L}}{\partial w}
  $$



With momentum, introduce a velocity term ($v_t$):

  $$
  w_{t+1} = w_t - \eta v_{t+1},
  $$
where

 $$
  v_{t+1} = \beta v_t + \frac{\partial \mathcal{L}}
  {\partial w_t}
  $$

The new velocity is the previous velocity, scaled by the momentum coefficient, plus the current gradient.

This gives the optimiser momentum.

Momentum accelerates weight updates when gradients consistently point in the same direction and smooths the effect of sudden changes in gradient direction

In [ ]:
display(Image(filename='./02-mnist-example/images/momentum.gif'))

In [ ]:
# Before:
optimiser = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
)

In [ ]:
# After:
optimiser = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
)

There are tonnes of way more advanced optimisers out there!

# Pre-training

When you initialise a model, the learnable parameters are randomly initialised:

In [ ]:
BasicCNN(num_classes=10, image_height=32, image_width=32).classifier.bias

In [ ]:
BasicCNN(num_classes=10, image_height=32, image_width=32).classifier.bias

Take a model that has been trained on a related task (with usually much more data available than your task)

This is called the "pre-trained" model. 

We want to transfer what it has learned to our task.

This can be thought of as adaptation of a model from the pre-trained task to the desired task.

This is called "fine-tuning". We want to fine-tune the pre-trained model on our task.

Lets download a pre-trained CNN that was trained on 1.28 million training images to predict 1000 classes:

In [ ]:
from transformers import AutoModelForImageClassification

model_name = "google/mobilenet_v2_0.35_96"

model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=10,
    ignore_mismatched_sizes=True,    
)


Pre-trained models often come with the data processor that they were trained with

In [ ]:
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained(model_name)


In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from tqdm.auto import tqdm


dataset = load_dataset("uoft-cs/cifar10", split="train")
to_tensor = ToTensor()
dataset.set_transform(
    lambda batch: {
        "img": [to_tensor(image) for image in batch["img"]],
        "label": batch["label"],
    }
)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

for mini_batch in tqdm(dataloader):
    images = mini_batch["img"]
    labels = mini_batch["label"]

    inputs = processor(images=images, return_tensors="pt")

    images = inputs['pixel_values']  # inputs is a dictionary.

    output = model(images)

    # ...
    # ... loss and optimisation, etc.
    # ...

# Note that this will significantly slow down training as it is a 1.5 million parameter model training on CPU! 3 minutes per epoch approx. 